# Colab 04 - Evaluate

This notebook generates audio for the test split using a fine-tuned checkpoint and writes evaluation outputs to Google Drive.

By default it picks the EMA final weights if present, then the last saved weights, then the base model as a fallback.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, '/content/MMAudio')

from fine_tune.urban_paths import (
    AUDIO_SR,
    CLIP_SEQ_LEN,
    DEFAULT_STRIDE,
    DRIVE_ROOT,
    EVAL_DIR,
    EXT_WEIGHTS_DIR,
    LATENT_SEQ_LEN,
    LOG_DIR,
    MODE,
    MODEL,
    OUTPUT_DIR,
    REPO_ROOT,
    SYNC_SEQ_LEN,
    TEXT_LABEL,
    TEXT_SEQ_LEN,
    TSV_DIR,
    VIDEO_EXTENSIONS,
    WEIGHTS_DIR,
    ensure_drive_dirs,
    latent_dir_for_stride,
    split_video_dirs,
)

STRIDE = DEFAULT_STRIDE
SPLIT_VIDEO_DIRS = split_video_dirs(STRIDE)
LATENTS_ROOT = latent_dir_for_stride(STRIDE)
RUNS_ROOT = OUTPUT_DIR
EVAL_ROOT = EVAL_DIR

ensure_drive_dirs(STRIDE)
for split_dir in SPLIT_VIDEO_DIRS.values():
    split_dir.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_ROOT)
print('Drive root:', DRIVE_ROOT)
print('Stride:', STRIDE)
print('Mode/model:', MODE, MODEL)
print('Latents:', LATENTS_ROOT)
print('Outputs:', RUNS_ROOT)
print('Eval:', EVAL_ROOT)
print('Logs:', LOG_DIR)

In [ ]:
%cd /content/MMAudio

# Keep repo-local paths compatible with scripts that expect ./weights and ./ext_weights,
# while storing the actual large files in Google Drive.
def link_drive_dir(repo_name: str, drive_path: Path):
    repo_path = REPO_ROOT / repo_name
    drive_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        return
    if repo_path.exists():
        if repo_path.is_dir() and not any(repo_path.iterdir()):
            repo_path.rmdir()
        else:
            print(f'{repo_path} exists; linking missing files inside it instead.')
            return
    repo_path.symlink_to(drive_path, target_is_directory=True)
    print(f'Linked {repo_path} -> {drive_path}')


def link_drive_file(repo_rel: str, drive_path: Path):
    repo_path = REPO_ROOT / repo_rel
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        return
    if repo_path.exists():
        if not drive_path.exists():
            repo_path.replace(drive_path)
            print(f'Moved {repo_path} -> {drive_path}')
        else:
            repo_path.unlink()
    repo_path.symlink_to(drive_path)
    print(f'Linked {repo_path} -> {drive_path}')


link_drive_dir('weights', WEIGHTS_DIR)
link_drive_dir('ext_weights', EXT_WEIGHTS_DIR)

for rel, target in {
    'weights/mmaudio_small_16k.pth': WEIGHTS_DIR / 'mmaudio_small_16k.pth',
    'ext_weights/v1-16.pth': EXT_WEIGHTS_DIR / 'v1-16.pth',
    'ext_weights/synchformer_state_dict.pth': EXT_WEIGHTS_DIR / 'synchformer_state_dict.pth',
    'ext_weights/best_netG.pt': EXT_WEIGHTS_DIR / 'best_netG.pt',
    'ext_weights/empty_string.pth': EXT_WEIGHTS_DIR / 'empty_string.pth',
}.items():
    link_drive_file(rel, target)

In [ ]:
%cd /content/MMAudio

EXP_ID = 'urban_ft_colab_small16k'
RUN_DIR = RUNS_ROOT / EXP_ID
GENERATED_DIR = EVAL_ROOT / EXP_ID / 'generated_audio'
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

candidate_weights = [
    RUN_DIR / f'{EXP_ID}_ema_final.pth',
    RUN_DIR / f'{EXP_ID}_last.pth',
    RUN_DIR / f'{EXP_ID}_shadow.pth',
    WEIGHTS_DIR / 'mmaudio_small_16k.pth',
]

WEIGHTS_PATH = next((p for p in candidate_weights if p.exists()), None)
if WEIGHTS_PATH is None:
    raise FileNotFoundError('No checkpoint or base weights found.')

print('Weights:', WEIGHTS_PATH)
print('Output:', GENERATED_DIR)

In [ ]:
%cd /content/MMAudio

import subprocess

TEST_TSV = TSV_DIR / 'video_test_ft.tsv'
TEST_VIDEO_ROOT = SPLIT_VIDEO_DIRS['test']

if not TEST_TSV.exists():
    raise FileNotFoundError(TEST_TSV)
if not TEST_VIDEO_ROOT.exists():
    raise FileNotFoundError(TEST_VIDEO_ROOT)

cmd = [
    'python', 'batch_eval_custom_ckpt.py',
    '--variant', MODEL,
    '--weights', str(WEIGHTS_PATH),
    '--test_tsv', str(TEST_TSV),
    '--video_root', str(TEST_VIDEO_ROOT),
    '--output_dir', str(GENERATED_DIR),
    '--duration', '8.0',
    '--seed', '42',
    '--cfg_strength', '4.5',
    '--num_steps', '25',
    '--skip_video_composite',
]

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
%cd /content/MMAudio

# Optional onset evaluation. Put ground-truth onset txt files here if you have them.
GT_ONSET_DIR = EVAL_ROOT / 'gt_onsets'

if GT_ONSET_DIR.exists():
    cmd = [
        'python', 'eval_onsets.py',
        '--input_dir', str(GENERATED_DIR),
        '--gt_dir', str(GT_ONSET_DIR),
        '--delta', '0.3',
    ]
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f'Skipping onset metrics; ground-truth directory does not exist: {GT_ONSET_DIR}')

In [ ]:
import json
from pathlib import Path

wav_files = sorted(GENERATED_DIR.glob('*.wav'))
mp4_files = sorted(GENERATED_DIR.glob('*.mp4'))
manifest = {
    'exp_id': EXP_ID,
    'weights': str(WEIGHTS_PATH),
    'generated_dir': str(GENERATED_DIR),
    'num_wav': len(wav_files),
    'num_mp4': len(mp4_files),
    'wav_files': [p.name for p in wav_files],
    'mp4_files': [p.name for p in mp4_files],
}
manifest_path = EVAL_ROOT / EXP_ID / 'evaluation_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

print(json.dumps({k: v for k, v in manifest.items() if k not in {'wav_files', 'mp4_files'}}, indent=2))
print('Manifest:', manifest_path)
if (GENERATED_DIR / 'eval_results.txt').exists():
    print((GENERATED_DIR / 'eval_results.txt').read_text())